# 上 上傳時間標籤

In [ ]:
# 上傳的時間格式 timezone 轉換本地時間
from pymongo import MongoClient
from datetime import datetime
from pytz import timezone # 需要安裝 pytz 套件

class MongoUploader:
    def __init__(self, connection_str, database, collection):
        self.client = MongoClient(connection_str)
        self.db = self.client[database]
        self.collection = self.db[collection]

    def add_upload_time(self):
        pipeline = [
            {
                "$addFields": {
                    "upload_time": {"$toDate": "$_id"}
                }
            }
        ]
        results = list(self.collection.aggregate(pipeline))
        
        for result in results:
            # 將 upload_time 轉成 UTC 時間
            utc_time = result["upload_time"].replace(tzinfo=timezone("UTC"))
            # 將 UTC 時間轉成本地時間 (台北)
            local_time = utc_time.astimezone(timezone("Asia/Taipei"))
            # 更新資料庫中的 upload_time 為本地時間
            self.collection.update_one(
                {"_id": result["_id"]},
                {"$set": {"upload_time": local_time}}
            )
            print(local_time)
        return results

# 使用範例
connection_str = "mongodb://localhost:27017"
database = "test"
collection = "test"

uploader = MongoUploader(connection_str, database, collection)
results=uploader.add_upload_time()
for result in results:
    print(result)


# 將每一筆資料插入upload_time和Heat的key value對

In [ ]:
# 最原本(只有pipeline和update_one)
from pymongo import MongoClient
import random
class MongoUploader:
    def __init__(self, connection_str, database, collection):
        self.client = MongoClient(connection_str)
        self.db = self.client[database]
        self.collection = self.db[collection]

    def add_upload_time(self):
        pipeline = [
            {
                "$addFields": {
                    "upload_time": {"$toDate": "$_id"}
                }
            }
        ]
        results = list(self.collection.aggregate(pipeline))
        
        for result in results:
            self.collection.update_one(
                {"_id": result["_id"]},
                {"$set": {"upload_time": result["upload_time"]}}
            )
        return results
    def add_heat(self):
        # 將 heat 數值插入每個文件
        results = self.collection.find()
        for result in results:
            self.collection.update_one(
                {"_id": result["_id"]},
                {"$set": {"Heat": random.randint(0, 10)}}
        )
    def top10(self, amount=10):
        results = self.collection.find().limit(amount)
        return [result for result in results]
'''
# 使用範例 (插入時間key value)
connection_str = "mongodb://localhost:27017"
database = "topic_Coll_Test"
collection = "original"

uploader = MongoUploader(connection_str, database, collection)
uploader.add_upload_time()
'''

# 使用範例 (插入Heat)
connection_str = "mongodb://localhost:27017"
database = "topic_Coll_Test"
collection = "original"
uploader = MongoUploader(connection_str, database, collection)
uploader.add_heat()
# check top 10 recorders
results=uploader.top10()
for result in results:
    print(result)


# 抓取資料並時間排序，資料最後回傳mongodb

In [ ]:
'''
from pymongo import MongoClient

# 抓取資料並時間排序
client = MongoClient('mongodb://localhost:27017/')
db = client.topic_Coll_Test
collection = db.original

cursor = collection.find().sort("local_upload_time", -1)

# 將查詢結果轉換成列表以便進行插入操作
documents = [doc for doc in cursor]
'''
# 資料最後回傳mongodb
class MongoWriter:
    def __init__(self, uri="mongodb://127.0.0.1:27017", database=None, collection=None):
        self.client = MongoClient(uri)
        self.db = self.client[database]
        self.collection = self.db[collection]
    
    def DeleAndInsert_data(self, data):
        try:
            self.collection.delete_many({})
            result = self.collection.insert_many(data)
            if result.acknowledged:
                return "Insertion successful"
            else:
                return "Insertion failed"
        except Exception as e:
            return "Insertion failed with error: {}".format(e)


In [ ]:
from pymongo import MongoClient

# 抓取資料並時間排序
client = MongoClient('mongodb://localhost:27017/')
db = client.topic_Coll_Test
collection = db.original

cursor = collection.find().sort("upload_time", -1)
# -1 照理來說應該是 大到小(時間晚的在前，時間早的在後)，結果輸出剛好相反

# 將查詢結果轉換成列表以便進行插入操作
documents = [doc for doc in cursor]# 這樣就拿到資料了(整個collection以list呈現)

# 新增collection的資料使用範例(先dalete再insert)
writer = MongoWriter(database="topic_collection", collection="NewEvent")
result = writer.DeleAndInsert_data(documents)# documents是已經排序的時間資料
print(result)
writer.client.close()

## 按照Heat排序，回傳Mongodb的指定位置

In [ ]:
from pymongo import MongoClient

# 抓取資料並時間排序
client = MongoClient('mongodb://localhost:27017/')
db = client.topic_Coll_Test
collection = db.original

cursor = collection.find().sort("Heat", -1)# -1是大到小

# 將查詢結果轉換成列表以便進行插入操作
documents = [doc for doc in cursor]

# 新增collection的資料使用範例(先dalete再insert)
writer = MongoWriter(database="topic_collection", collection="HeatEvent")
result = writer.DeleAndInsert_data(documents)# documents是已經排序的時間資料
print(result)
writer.client.close()

## 製造測資

In [ ]:
from pymongo import MongoClient

# 连接到MongoDB数据库
client = MongoClient('mongodb://localhost:27017/')

# 获取test集合
db = client['test']
collection = db['test']

# 添加示例文档
docs = [
    {"name": "Alice", "age": 25, "city": "New York"},
    {"name": "Bob", "age": 30, "city": "Los Angeles"},
    {"name": "Charlie", "age": 35, "city": "Chicago"},
    {"name": "David", "age": 40, "city": "Houston"},
    {"name": "Eva", "age": 45, "city": "Philadelphia"},
    {"name": "Frank", "age": 50, "city": "Phoenix"},
    {"name": "Grace", "age": 55, "city": "San Antonio"},
    {"name": "Henry", "age": 60, "city": "San Diego"},
    {"name": "Ivy", "age": 65, "city": "Dallas"},
    {"name": "Jack", "age": 70, "city": "Austin"}
]
result = collection.insert_many(docs)

# 打印插入结果
print(result.inserted_ids)